# Database preparation
This code is for preparing the database for stage II. We provided this code as a sample to generate the database by the training dataset. You can create the database from your source. 

In [ ]:
import os
import faiss
import pandas as pd
import numpy as np
from utils.earth_computation import distance_to_coastline
from pathlib import Path

root = Path("/path/to") # [TODO] path to data source
file_path = root / 'dataset_final.csv'  
df = pd.read_csv(file_path)
 
# select the columns we need
selected_columns = ['mmsi', 'lat', 'lon','sog', 'cog', 'next_code', 'next_lat', 'next_lon', "coastline_distance"]
df_selected = df[selected_columns]
grouped = df_selected.groupby('mmsi')

dest_rec = {}

for mmsi, group_data in grouped:
    sp = 0
    for i in range(len(group_data)):
        if group_data.iloc[i]['next_code'] != group_data.iloc[i-1]['next_code']:
            port_code = group_data.iloc[i-1]['next_code']
            if port_code not in dest_rec:
                dest_rec[port_code] = []
            dest_rec[port_code].append(group_data.iloc[sp:i])
            sp = i

dict_keys(['CNYTN', 'CAHFX', 'USNYK', 'USNFK', 'USSAV', 'USCHA', 'GOOD_HOPE', 'MALACCA_STRAIT', 'MYPKE', 'SGSGP', 'SINGAPORE_STRAIT', 'THLCB', 'VNCMP', 'TAIWAN_STRAIT', 'EKATERINA_BAY', 'USLSA', 'USOAK', 'ENGLISH_CHANNEL', 'TSINGYI_STRAIT', 'KOREA_STRAIT', 'CNZOS', 'CNMWN', 'LKCOL', 'KRENITSYN_STRAIT', 'BASHI_STRAIT', 'TWKAO', 'MXLCS', 'JPKUR', 'HKHKG', 'CNXIA', 'JPYOK', 'CNFUN', 'CNYSN', 'CNQIN', 'KRBUS', 'MXMAN', 'ECPSJ', 'PABAL', 'ITGIT', 'MATAN', 'GIBRALTAR_STRAIT', 'CNNBO', 'CNNSA', 'ESVAL', 'ESBAR', 'CNGUA', 'USLGB', 'CAPE_HAITIEN', 'INNYY', 'OMSAL', 'MYTPP', 'DOVER_STRAIT', 'GBFEL', 'DEWVN', 'DEHAM', 'NLROT', 'BEANT', 'GBLGP', 'NASSAU', 'YUCATAN_CHANNEL', 'BARINGTANG_STRAIT', 'CNTJN', 'CNDAL', 'CNSHE', 'EGAKI', 'DARDANELLES_STRAIT', 'TRYAR', 'TRBAM', 'TRTEK', 'ILASD', 'PROLIV_KRUZENSHTERNA', 'SOYA_STRAIT', 'AKASHI_STRAIT', 'FEMER_STRAIT', 'USSEA', 'CAPRR', 'NORTH_CHANNEL', 'CNCGM', 'PANAMA_CANAL', 'NADEZHDY_STRAIT', 'CNYPG', 'QIONGZHOU_STRAIT', 'EVREINOV_STRAIT', 'PACOL', 'USBAL

In [7]:
print(dest_rec.keys())
print(len(dest_rec["CAHFX"]))
print(dest_rec["CAHFX"])

dict_keys(['CNYTN', 'CAHFX', 'USNYK', 'USNFK', 'USSAV', 'USCHA', 'GOOD_HOPE', 'MALACCA_STRAIT', 'MYPKE', 'SGSGP', 'SINGAPORE_STRAIT', 'THLCB', 'VNCMP', 'TAIWAN_STRAIT', 'EKATERINA_BAY', 'USLSA', 'USOAK', 'ENGLISH_CHANNEL', 'TSINGYI_STRAIT', 'KOREA_STRAIT', 'CNZOS', 'CNMWN', 'LKCOL', 'KRENITSYN_STRAIT', 'BASHI_STRAIT', 'TWKAO', 'MXLCS', 'JPKUR', 'HKHKG', 'CNXIA', 'JPYOK', 'CNFUN', 'CNYSN', 'CNQIN', 'KRBUS', 'MXMAN', 'ECPSJ', 'PABAL', 'ITGIT', 'MATAN', 'GIBRALTAR_STRAIT', 'CNNBO', 'CNNSA', 'ESVAL', 'ESBAR', 'CNGUA', 'USLGB', 'CAPE_HAITIEN', 'INNYY', 'OMSAL', 'MYTPP', 'DOVER_STRAIT', 'GBFEL', 'DEWVN', 'DEHAM', 'NLROT', 'BEANT', 'GBLGP', 'NASSAU', 'YUCATAN_CHANNEL', 'BARINGTANG_STRAIT', 'CNTJN', 'CNDAL', 'CNSHE', 'EGAKI', 'DARDANELLES_STRAIT', 'TRYAR', 'TRBAM', 'TRTEK', 'ILASD', 'PROLIV_KRUZENSHTERNA', 'SOYA_STRAIT', 'AKASHI_STRAIT', 'FEMER_STRAIT', 'USSEA', 'CAPRR', 'NORTH_CHANNEL', 'CNCGM', 'PANAMA_CANAL', 'NADEZHDY_STRAIT', 'CNYPG', 'QIONGZHOU_STRAIT', 'EVREINOV_STRAIT', 'PACOL', 'USBAL

In [22]:
print("name of dest:", dest_rec.keys())
print("num of dest: ", len(dest_rec.keys()))
# print(len(dest_rec["MALACCA_STRAIT"]))
# print(dest_rec["MALACCA_STRAIT"])
avg, cnt = 0, 0
for dest in dest_rec.keys():
    i = 0
    while i < len(dest_rec[dest]):
        # print(type(dest_rec[dest][i]))
        traj = dest_rec[dest][i]
        if len(traj) < 500:
            # print(f"dest {dest} has traj with len {len(traj)}")
            dest_rec[dest].pop(i)
            print(f"remove traj with len {len(traj)}")
        else:
            i += 1
    avg += len(dest_rec[dest])
    cnt += 1
# print("avg num of traj per dest: ", avg / cnt)
avg, cnt = 0, 0
avg1, cnt1 = 0, 0
for dest in dest_rec.keys():
    # print(dest,": ", len(dest_rec[dest]), sep = "")
    for i in range(len(dest_rec[dest])):
        traj = dest_rec[dest][i]
        assert len(traj) >= 500
        avg += len(dest_rec[dest][i])
        cnt += 1
    avg1 += len(dest_rec[dest])
    cnt1 += 1
print("avg dist of traj per dest: ", avg / cnt)
print("avg num of traj per dest: ", avg1 / cnt1)


# print(dest_rec["CAHFX"][5].tail(10))

name of dest: dict_keys(['CAHFX', 'USNYK', 'USNFK', 'USSAV', 'USCHA', 'GOOD_HOPE', 'MALACCA_STRAIT', 'MYPKE', 'SGSGP', 'SINGAPORE_STRAIT', 'THLCB', 'VNCMP', 'CNYTN', 'TAIWAN_STRAIT', 'EKATERINA_BAY', 'USLSA', 'USOAK', 'ENGLISH_CHANNEL', 'TSINGYI_STRAIT', 'KOREA_STRAIT', 'CNZOS', 'CNMWN', 'LKCOL', 'KRENITSYN_STRAIT', 'BASHI_STRAIT', 'TWKAO', 'JPKUR', 'HKHKG', 'CNXIA', 'JPYOK', 'CNFUN', 'CNYSN', 'CNQIN', 'KRBUS', 'MXMAN', 'MXLCS', 'ECPSJ', 'PABAL', 'ITGIT', 'MATAN', 'GIBRALTAR_STRAIT', 'CNNBO', 'CNNSA', 'ESVAL', 'ESBAR', 'CNGUA', 'USLGB', 'CAPE_HAITIEN', 'INNYY', 'OMSAL', 'MYTPP', 'DOVER_STRAIT', 'GBFEL', 'DEWVN', 'DEHAM', 'NLROT', 'BEANT', 'GBLGP', 'NASSAU', 'YUCATAN_CHANNEL', 'BARINGTANG_STRAIT', 'CNTJN', 'CNDAL', 'CNSHE', 'EGAKI', 'DARDANELLES_STRAIT', 'TRYAR', 'TRBAM', 'TRTEK', 'ILASD', 'PROLIV_KRUZENSHTERNA', 'SOYA_STRAIT', 'AKASHI_STRAIT', 'FEMER_STRAIT', 'USSEA', 'CAPRR', 'NORTH_CHANNEL', 'CNCGM', 'PANAMA_CANAL', 'NADEZHDY_STRAIT', 'CNYPG', 'QIONGZHOU_STRAIT', 'EVREINOV_STRAIT', '

In [ ]:
from utils.process import deg_to_rad, deg_to_vec
import numpy as np

def make_windows(df, stride):

    samples = []
    required_columns = ["mmsi", "lat", "lon", "sog", "cog", "coastline_distance"]
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"column {col} not exists")
        
    # group by mmsi
    coord = df[["lat", "lon", "sog", "cog", "coastline_distance", "next_lat", "next_lon"]].values

    lat = deg_to_rad(coord[:,0]).astype(np.float32)
    lon = deg_to_rad(coord[:,1]).astype(np.float32)
    sog = coord[:,2].astype(np.float32)
    cog = deg_to_vec(coord[:,3]).astype(np.float32)
    dist = coord[:,4].astype(np.float32)

    target_lat = deg_to_rad(coord[:,5]).astype(np.float32)
    target_lon = deg_to_rad(coord[:,6]).astype(np.float32)

    N = len(coord)
    total_len = 288
    cnt = 0
    
    # sliding the windows
    for s in range(0, N-total_len+1, stride):
        lat_window = lat[s:s+total_len, None]
        lon_window = lon[s:s+total_len, None]
        sog_window = sog[s:s+total_len, None]
        cog_window = cog[s:s+total_len]
        dist_window = dist[s:s+total_len]

        target_lat_window = target_lat[s:s+total_len, None]
        target_lon_window = target_lon[s:s+total_len, None]
        
        if sog_window.min() > 1.5 and dist_window.min() > 80000:
            merging = np.concatenate((lat_window, lon_window, sog_window*cog_window, target_lat_window, target_lon_window), axis=1)
            samples.append(merging)
        else:
            cnt+=1
    
    return samples

In [ ]:
from random import sample

num_traj_per_dest = 50 # [TODO] tune this hyperparameter
final_rec = []
stride = 1
deposited_port = 0

for dest in dest_rec.keys():
    samples = []
    # num = 0
    for traj in dest_rec[dest]:
        samples += make_windows(traj, stride)
    if len(samples) >= num_traj_per_dest:
        final_rec += sample(samples, num_traj_per_dest)
        # print(f"dest {dest} could generate {len(samples)} windows, but {num_traj_per_dest} windows are sampled")
    else:
        print(f"dest {dest} could generate {len(samples)} windows, but {num_traj_per_dest} windows are required")
        deposited_port += 1
        # final_rec += samples

print(len(final_rec))
print(f"deposited {deposited_port} ports")


dest USNFK could generate 0 windows, but 50 windows are required
dest USSAV could generate 0 windows, but 50 windows are required
dest USCHA could generate 0 windows, but 50 windows are required
dest MYPKE could generate 0 windows, but 50 windows are required
dest SGSGP could generate 0 windows, but 50 windows are required
dest USOAK could generate 0 windows, but 50 windows are required
dest CNMWN could generate 0 windows, but 50 windows are required
dest JPKUR could generate 0 windows, but 50 windows are required
dest HKHKG could generate 17 windows, but 50 windows are required
dest CNFUN could generate 0 windows, but 50 windows are required
dest KRBUS could generate 0 windows, but 50 windows are required
dest ITGIT could generate 0 windows, but 50 windows are required
dest MATAN could generate 0 windows, but 50 windows are required
dest ESVAL could generate 0 windows, but 50 windows are required
dest ESBAR could generate 0 windows, but 50 windows are required
dest CNGUA could generat

In [65]:
dataList = np.array(final_rec)
print(dataList.shape)
np.save('enrolled_trajectory.npy', dataList)


(2800, 288, 6)
